# 02 · Limpieza, Codificación y Normalización

Prepara `features_municipio_cultivo` para el modelado:

1. **Limpieza** — tipos, duplicados, imputación de faltantes.
2. **Codificación** — variables categóricas (`cultivo`, `fase_fenologica`).
3. **Normalización** — min-max robusto (p1–p99), reutilizando el código de producción
   `src/features/normalize.py` para replicar exactamente lo que hace el IRA.


In [ ]:
# Arranque: localizar la raiz del repo y la base de datos DuckDB
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import duckdb
import numpy as np
import pandas as pd

from config import config

DB_PATH = ROOT / config.duckdb_path
DB_OK = DB_PATH.exists()

print("Repo :", ROOT)
print("BD   :", DB_PATH)
print("Estado:", "OK, base encontrada" if DB_OK else "NO existe -> corre `make pipeline` antes de este notebook")


def con():
    if not DB_OK:
        raise FileNotFoundError(
            f"No existe {DB_PATH}. Ejecuta el pipeline:  python scripts/run.py ingest && features && risk"
        )
    c = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        c.execute("INSTALL spatial; LOAD spatial;")
    except Exception:
        pass
    return c


def q(sql: str) -> pd.DataFrame:
    with con() as c:
        return c.execute(sql).df()


In [ ]:
fmc = q("SELECT * FROM features_municipio_cultivo")
print("Dimensiones:", fmc.shape)
fmc.dtypes.to_frame("dtype")


## 1. Limpieza básica


In [ ]:
# Duplicados por la clave primaria logica
clave = ["codigo_municipio", "cultivo", "periodo"]
dups = fmc.duplicated(subset=clave).sum()
print("Filas duplicadas por clave (municipio, cultivo, periodo):", dups)

# Faltantes por columna
faltantes = fmc.isna().sum()
faltantes[faltantes > 0].sort_values(ascending=False).to_frame("n_nulos")


In [ ]:
# Imputacion: numericas -> 0 (convenio del pipeline: ausencia de senal = 0 tras normalizar)
#             categoricas -> "desconocido"
from pandas.api.types import is_numeric_dtype

df = fmc.copy()
num_cols = [c for c in df.columns if is_numeric_dtype(df[c]) and c not in ("codigo_municipio",)]
cat_cols = [c for c in df.select_dtypes(include="object").columns if c not in clave]

df[num_cols] = df[num_cols].fillna(0.0)
for c in cat_cols:
    df[c] = df[c].fillna("desconocido")

print("Nulos restantes:", int(df.isna().sum().sum()))
print("Columnas categoricas detectadas:", cat_cols)


## 2. Codificación de categóricas

`fase_fenologica` es ordinal (etapa del cultivo); `cultivo` es nominal de alta cardinalidad.


In [ ]:
# Codificacion de cultivo con codigos enteros (label encoding) para inspeccion
if "cultivo" in df.columns:
    df["cultivo_cod"] = df["cultivo"].astype("category").cat.codes
    print("Cultivos distintos:", df["cultivo"].nunique())
    display(df[["cultivo", "cultivo_cod"]].drop_duplicates().head(10))

# fase_fenologica: si viene como texto, mapear a orden logico; si es numerica, dejarla
if "fase_fenologica" in df.columns and df["fase_fenologica"].dtype == object:
    orden = {"desconocido": 0, "siembra": 1, "desarrollo": 2, "floracion": 3,
             "llenado": 4, "cosecha": 5}
    df["fase_fenologica_cod"] = (
        df["fase_fenologica"].str.lower().map(orden).fillna(0).astype(int)
    )
    display(df[["fase_fenologica", "fase_fenologica_cod"]].drop_duplicates())


## 3. Normalización robusta (p1–p99)

Reutilizamos `src/features/normalize.py` — la **misma** función que usa el motor del IRA,
para que el análisis sea consistente con producción. Escala cada variable a `[0, 1]`
recortando en los percentiles 1 y 99 (robusto a outliers).


In [ ]:
from src.features.normalize import ALL_FEATURE_COLS, normalize

print("Variables que normaliza el IRA (", len(ALL_FEATURE_COLS), "):")
for c in ALL_FEATURE_COLS:
    print("  -", c)


In [ ]:
# Antes: rangos crudos de una muestra de variables
muestra = [c for c in ["precip_acum_30d", "tmax_media_7d", "area_sembrada",
                       "rendimiento_promedio", "nbi_total"] if c in df.columns]
print("ANTES (crudo):")
display(df[muestra].describe().T[["min", "mean", "max"]])

df_norm = normalize(df)

print("DESPUES (normalizado p1-p99):")
display(df_norm[muestra].describe().T[["min", "mean", "max"]])


In [ ]:
# Verificacion: todas las features normalizadas deben quedar en [0, 1]
fuera_rango = {
    c: (float(df_norm[c].min()), float(df_norm[c].max()))
    for c in ALL_FEATURE_COLS
    if c in df_norm and (df_norm[c].min() < -1e-9 or df_norm[c].max() > 1 + 1e-9)
}
print("Columnas fuera de [0,1]:", fuera_rango if fuera_rango else "ninguna (OK)")


## 4. Guardar dataset procesado


In [ ]:
# Guardamos el resultado limpio+normalizado para los notebooks siguientes
salida = ROOT / "notebooks" / "_data_procesada.parquet"
cols_guardar = clave + [c for c in ALL_FEATURE_COLS if c in df_norm.columns]
df_norm[cols_guardar].to_parquet(salida, index=False)
print("Guardado:", salida, "|", df_norm[cols_guardar].shape)


---
### Resumen

- Imputamos faltantes (numéricas → 0, categóricas → "desconocido") siguiendo el convenio del pipeline.
- Codificamos `cultivo` (label) y `fase_fenologica` (ordinal).
- Normalizamos las 26 variables con el mismo min-max robusto p1–p99 de producción → todas en `[0, 1]`.
- El dataset procesado queda en `notebooks/_data_procesada.parquet`.

Siguiente paso → **`03_analisis_descriptivo.ipynb`**.
